In [1]:

from ogmios.model import get_hifigan

In [3]:
model = get_hifigan(checkpoint="LJ_V2/generator_v2",
                    infer_device="cpu")

Removing weight norm...


/home/hadware/Code/ogmios-workbench/efficientspeech/venv/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [4]:



def get_module_size(module, prefix=""):
    """Récursivement calcule la taille de chaque sous-module d'un modèle PyTorch en mégaoctets."""
    submodule_sizes = {}
    total_size = 0

    # Parcours des sous-modules
    for name, submodule in module.named_children():
        # Calcul de la taille des paramètres du sous-module en octets
        submodule_size, sizes = get_module_size(submodule, prefix=f"{prefix}{name}.")
        submodule_sizes.update(sizes)
        total_size += submodule_size

    # Taille du module actuel
    module_size = sum(p.numel() * p.element_size() for p in module.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in module.buffers())
    current_size = module_size + buffer_size  # Taille totale en octets pour ce module
    total_size += current_size

    # Enregistrer la taille du module actuel
    submodule_sizes[f"{prefix}{module.__class__.__name__}"] = current_size / (1024 * 1024)  # Conversion en mégaoctets

    return total_size, submodule_sizes


def print_module_sizes(module):
    """Affiche les tailles des sous-modules en mégaoctets."""
    _, sizes = get_module_size(module)
    print("\nTaille des sous-modules (en MB) :\n")
    for name, size in sizes.items():
        print(f"{name}: {size:.2f} MB")


# Affiche les tailles des sous-modules
print_module_sizes(model)



Taille des sous-modules (en MB) :

conv_pre.Conv1d: 0.27 MB
ups.0.ConvTranspose1d: 0.50 MB
ups.1.ConvTranspose1d: 0.13 MB
ups.2.ConvTranspose1d: 0.01 MB
ups.3.ConvTranspose1d: 0.00 MB
ups.ModuleList: 0.64 MB
resblocks.0.convs1.0.Conv1d: 0.05 MB
resblocks.0.convs1.1.Conv1d: 0.05 MB
resblocks.0.convs1.2.Conv1d: 0.05 MB
resblocks.0.convs1.ModuleList: 0.14 MB
resblocks.0.convs2.0.Conv1d: 0.05 MB
resblocks.0.convs2.1.Conv1d: 0.05 MB
resblocks.0.convs2.2.Conv1d: 0.05 MB
resblocks.0.convs2.ModuleList: 0.14 MB
resblocks.0.ResBlock1: 0.28 MB
resblocks.1.convs1.0.Conv1d: 0.11 MB
resblocks.1.convs1.1.Conv1d: 0.11 MB
resblocks.1.convs1.2.Conv1d: 0.11 MB
resblocks.1.convs1.ModuleList: 0.33 MB
resblocks.1.convs2.0.Conv1d: 0.11 MB
resblocks.1.convs2.1.Conv1d: 0.11 MB
resblocks.1.convs2.2.Conv1d: 0.11 MB
resblocks.1.convs2.ModuleList: 0.33 MB
resblocks.1.ResBlock1: 0.66 MB
resblocks.2.convs1.0.Conv1d: 0.17 MB
resblocks.2.convs1.1.Conv1d: 0.17 MB
resblocks.2.convs1.2.Conv1d: 0.17 MB
resblocks.2.convs1

In [14]:
model.resblocks[0].convs1[0].weight.shape

torch.Size([64, 64, 3])